# Drave — le calcul réparti, vu de Colab

Ce notebook fait tourner le **pilote** de Drave sur Colab : il calcule les
premières couches du modèle, envoie le flux d'activations à votre machine qui
calcule les dernières, et compare la perplexité obtenue à celle du même modèle
calculé entièrement ici.

**Le critère de succès est la fidélité, pas la vitesse** : `écart : 0.00e0` en
mode dense (le transport f32 est sans perte). Comptez ~8 jetons/s à 60 ms
d'aller-retour, contre ~60 en local — le découpage sert à faire tenir un modèle
qui ne tient pas sur une machine.

**Avant de lancer** : sur votre machine, `tools/preparer_colab.sh` a produit
`drave.bundle` (≈5 Mo) et vous avez noté le SHA-256 du modèle. Uploadez le bundle
et le GGUF ici (ou mettez-les sur votre Drive).

## 1. L'environnement

In [ ]:
import os, shutil, subprocess, platform
print(platform.platform())
print("cœurs :", os.cpu_count())
with open('/proc/meminfo') as f:
    for l in f:
        if l.startswith('MemTotal'): print("RAM :", l.split()[1], "ko")
print("GPU :", subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                             capture_output=True, text=True).stdout.strip() or "aucun")
print("cargo :", shutil.which('cargo') or "absent — la cellule suivante l'installe")
print("taille du disque libre :", subprocess.run(['df','-h','/content'], capture_output=True,
                                                text=True).stdout.splitlines()[-1])

## 2. La chaîne Rust (seulement si `cargo` manque)

In [ ]:
import shutil, subprocess
if not shutil.which('cargo'):
    subprocess.run('curl --proto \'=https\' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y',
                   shell=True, check=True)
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
print(subprocess.run(['cargo','--version'], capture_output=True, text=True).stdout.strip())

## 3. Le code

Le bundle git contient **tout l'historique** : c'est le dépôt, en un fichier.
Adaptez le chemin si vous l'avez mis sur Drive.

In [ ]:
BUNDLE = '/content/drave.bundle'      # ou '/content/drive/MyDrive/drave.bundle'
if not os.path.exists('/content/drave'):
    subprocess.run(['git','clone', BUNDLE, '/content/drave'], check=True)
print(subprocess.run(['git','-C','/content/drave','log','--oneline','-1'],
                     capture_output=True, text=True).stdout.strip())

## 4. Le modèle — le MÊME fichier que le nœud moteur

Le protocole ne transporte pas la dimension : les deux nœuds doivent charger le
même GGUF. Vérifiez l'empreinte :

In [ ]:
MODELE = '/content/Qwen2.5-0.5B-Q8_0.gguf'   # ou sur Drive
ATTENDU = '25130a98aa782284a7dabea0'         # les 24 premiers caractères, imprimés par preparer_colab.sh
import hashlib, os
if not os.path.exists(MODELE):
    from google.colab import files; files.upload()
h = hashlib.sha256(open(MODELE,'rb').read()).hexdigest()[:24]
print(MODELE, os.path.getsize(MODELE)//2**20, "Mo · sha256", h)
print("MÊME MODÈLE QUE LE NŒUD MOTEUR" if h == ATTENDU else "⚠ MODÈLE DIFFÉRENT — le calcul divergera")

## 5. Compilation (≈1 min sur Colab)

In [ ]:
%%bash
cd /content/drave/drave-core && cargo build --release 2>&1 | tail -3
ls -la target/release/drave

## 6. Le banc réparti

Sur votre machine, le nœud moteur écoute déjà (derrière un tunnel TCP) :

```
drave models/Qwen2.5-0.5B-Q8_0.gguf --flux-serveur 0.0.0.0:5599 --ctx 256
# puis le tunnel : ngrok tcp 5599 · ou tailscale · ou ssh -R 5599:localhost:5599 vps
```

Remplacez `HOTE:PORT` par ce que le tunnel vous a donné.

## 6. Un texte réel pour une perplexité qui veut dire quelque chose

Sans fichier d'identifiants, le pilote retombe sur des identifiants
synthétiques : l'ÉCART reste valide (c'est le critère), mais la perplexité se
compte en milliers et ne dit rien. Cette cellule tokenise un paragraphe avec le
tokenizer du MODÈLE et s'en sert.


In [ ]:
TEXTE = '''La riviere descendait de la montagne en trois cascades, et le moulin,
au pied de la derniere, tournait depuis deux siecles. Quand l'eau baissait,
on entendait les meules grincer sur la pierre ; quand elle montait, le bruit
de l'eau couvrait tout, et les meuniers parlaient plus fort sans s'en rendre
compte. Le village entier vivait au rythme de cette chute.'''
open('/content/texte.txt','w').write(TEXTE)
import subprocess, re
subprocess.run(['/content/drave/drave-core/target/release/drave', MODELE,
                '--tokenize', '--fichier', '/content/texte.txt',
                '--sortie', '/content/ids.txt'], check=True)
# le tokenizer ecrit les identifiants separes par des espaces, des virgules ou
# des retours a la ligne selon la version : on lit tout ce qui est entier
IDS_TEXTE = [int(x) for x in re.findall(rb'\d+', open('/content/ids.txt','rb').read())]
print(len(IDS_TEXTE), 'identifiants · premiers :', IDS_TEXTE[:8])

## 7. Lire le résultat

```
  référence locale  : perplexité ~20        ← le modèle entier, calculé sur Colab
  deux nœuds        : perplexité ~20        ← 12 couches ici, 12 chez vous
  écart             : 0.00e0                 ← LE critère
  transport         : LOTS de 64 jetons · 229415 octets par lot · 4 lots
  temps             : 1606.6 ms par LOT en local · 1567.3 ms à attendre le pair
```

* `écart : 0.00e0` → le découpage est fidèle au bit près. C'est réussi.
* la perplexité attendue est de l'ordre de **20** (mesuré : 19,8967 sur le
  paragraphe de la cellule 6, 101 identifiants). Avec des identifiants
  synthétiques elle se compterait en milliers : c'est pour cela que la cellule 6
  tokenise un vrai texte.
* `--lot 64` regroupe 64 jetons par aller-retour : c'est le préremplissage
  réparti. Sans lui, un flux par jeton, et la latence est payée 256 fois.
* un écart non nul en mode dense → vérifier d'abord que l'empreinte du modèle est
  la même des deux côtés, puis la version du protocole (un pair incompatible est
  refusé en la nommant).

**Dépannage** : si le tunnel réintroduit de la mise en tampon, l'attente explose
(Nagle a coûté 6× le temps de calcul une fois) ; `TCP_NODELAY` est posé par le
moteur, mais un proxy peut l'annuler. Si `--flux-serveur` n'écoute pas,
`ss -ltn | grep 5599` sur votre machine.